In [ ]:
%pip install -q rapidfuzz

In [ ]:
import os
import re
import math
import unicodedata
from typing import Dict, List, Tuple, Set

import numpy as np
import pandas as pd

from datasets import load_dataset
from rapidfuzz import fuzz
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

TEST_CSV = "/kaggle/input/datasets/sohatiy/check-val/check.txt"

OUT_ALL = "contamination_candidates_v2.csv"
OUT_SUSPICIOUS = "contamination_suspicious_only_v2.csv"
OUT_TOP_BY_TEST = "contamination_top_by_test_v2.csv"

HF_DATASET = "ycchen/Crystal-Math-Preview"
HF_CONFIG = "rewritten"
HF_SPLIT = "train"

PASS_RATE_THRESHOLD = 0.5

EMB_MODEL = os.getenv("EMB_MODEL", "BAAI/bge-m3")

BATCH_SIZE = int(os.getenv("BATCH_SIZE", "32"))

TOPK_EMBEDDING = 80
TOPK_MASKED_FUZZY = 80
TOPK_RAW_FUZZY = 40
TOPK_PER_TEST_TO_SAVE = 25

MIN_SAVE_SCORE = 0.52


def is_missing(x) -> bool:
    return x is None or (isinstance(x, float) and math.isnan(x))


def to_str(x) -> str:
    if is_missing(x):
        return ""
    return str(x)


def normalize_unicode(s: str) -> str:
    s = to_str(s)
    s = unicodedata.normalize("NFKC", s)

    replacements = {
        "“": '"',
        "”": '"',
        "«": '"',
        "»": '"',
        "‘": "'",
        "’": "'",
        "−": "-",
        "–": "-",
        "—": "-",
        "×": " x ",
        "·": " ",
        "…": "...",
    }

    for a, b in replacements.items():
        s = s.replace(a, b)

    return s


def normalize_math_text(s: str) -> str:
    s = normalize_unicode(s).lower()

    latex_replacements = {
        "\\times": " x ",
        "\\cdot": " ",
        "\\leq": " <= ",
        "\\geq": " >= ",
        "\\neq": " != ",
        "\\frac": " frac ",
        "\\sqrt": " sqrt ",
        "\\binom": " binom ",
        "\\mathbb": " ",
        "\\mathcal": " ",
        "\\boxed": " ",
        "\\left": " ",
        "\\right": " ",
    }

    for a, b in latex_replacements.items():
        s = s.replace(a, b)

    s = re.sub(r"\\[a-zA-Z]+", " ", s)
    s = re.sub(r"[{}$^_]", " ", s)
    s = re.sub(r"[^a-zа-яё0-9+\-*/=<>. ]+", " ", s, flags=re.IGNORECASE)

    s = re.sub(r"\s+", " ", s).strip()
    return s


def normalize_for_embedding(s: str) -> str:
    s = normalize_unicode(s).lower()
    s = s.replace("\\times", " times ")
    s = s.replace("\\cdot", " ")
    s = re.sub(r"[{}$]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def mask_numbers(s: str) -> str:
    s = normalize_math_text(s)
    s = re.sub(r"\d+(?:\.\d+)?", " NUM ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def compact_math_signature(s: str) -> str:
    s = mask_numbers(s)

    stop = {
        "the", "a", "an", "of", "in", "on", "to", "from", "for", "with", "such",
        "that", "there", "are", "is", "be", "can", "could", "what", "find",
        "maximum", "minimum", "possible", "value", "number",
        "let", "suppose", "known", "given",
        "и", "в", "на", "для", "что", "как", "найдите", "найти", "число",
        "значение", "максимальное", "минимальное", "возможное",
    }

    tokens = [t for t in s.split() if t not in stop]
    return " ".join(tokens)


def safe_col(df: pd.DataFrame, name: str) -> pd.Series:
    if name in df.columns:
        return df[name].fillna("").astype(str)
    return pd.Series([""] * len(df), index=df.index)


def extract_numbers(s: str) -> List[str]:
    s = normalize_math_text(s)
    return re.findall(r"(?<![a-zа-яё])\d+(?:\.\d+)?", s)


def normalize_number_token(x: str) -> str:
    try:
        v = float(x)
        if v.is_integer():
            return str(int(v))
        return str(v)
    except Exception:
        return x


def extract_rare_numbers(s: str) -> List[str]:
    nums = []
    for x in extract_numbers(s):
        x = normalize_number_token(x)
        try:
            v = float(x)
        except Exception:
            continue

        if v >= 20:
            nums.append(str(int(v)) if v.is_integer() else str(v))

    return nums


def number_overlap_features(test_text: str, train_text: str) -> Tuple[float, str, int]:
    a = set(extract_rare_numbers(test_text))
    b = set(extract_rare_numbers(train_text))

    if not a:
        return 0.0, "", 0

    common = sorted(a & b, key=lambda x: (-len(x), x))

    if len(common) == 1:
        only = common[0]
        if only.isdigit() and int(only) < 1000:
            return 0.0, only, 1

    score = len(common) / max(1, len(a))
    return score, ",".join(common), len(common)


def has_large_number_hit(test_text: str, train_text: str) -> bool:
    a = set(extract_rare_numbers(test_text))
    b = set(extract_rare_numbers(train_text))
    common = a & b

    for x in common:
        if x.isdigit() and int(x) >= 1000:
            return True

    return False


def raw_fuzzy(a: str, b: str) -> float:
    return fuzz.token_set_ratio(normalize_math_text(a), normalize_math_text(b)) / 100.0


def masked_fuzzy(a: str, b: str) -> float:
    return fuzz.token_set_ratio(mask_numbers(a), mask_numbers(b)) / 100.0


def signature_fuzzy(a: str, b: str) -> float:
    return fuzz.token_set_ratio(compact_math_signature(a), compact_math_signature(b)) / 100.0


def exact_normalized_match(a: str, b: str) -> bool:
    return normalize_math_text(a) == normalize_math_text(b)


def answer_match_weak_flag(a, b) -> bool:
    if is_missing(a) or is_missing(b):
        return False

    aa = normalize_math_text(str(a))
    bb = normalize_math_text(str(b))
    return aa != "" and aa == bb


def keyword_set(s: str) -> Set[str]:
    s = normalize_math_text(s)
    tokens = set(s.split())
    useful = set()

    keep_words = {
        "sum", "terms", "power", "powers", "non", "negative", "integer", "integers",
        "represented", "representation", "m", "n",
        "row", "column", "table", "swap", "adjacent", "permutation",
        "word", "words", "alphabet", "letters", "position",
        "ray", "rays", "space", "sphere", "points",
        "gcd", "greatest", "common", "divisor", "adjacent",
        "cube", "cubes", "block", "red", "blue", "prism",
        "continuous", "function", "chord", "graph", "length",
        "cities", "roads", "components", "expected", "cheapest",
        "segments", "endpoint", "label", "smallest",
        "сумма", "степеней", "степени", "неотрицательных", "целых",
        "строка", "столбец", "таблица", "перестановка", "поменять",
        "слово", "слова", "буквы", "алфавит", "позиция",
        "лучи", "луч", "пространстве", "сфера", "точки",
        "нод", "общий", "делитель", "соседних",
        "куб", "кубики", "призма", "красный", "синий",
    }

    for t in tokens:
        if t in keep_words:
            useful.add(t)

    return useful


def keyword_overlap_score(a: str, b: str) -> float:
    ka = keyword_set(a)
    kb = keyword_set(b)

    if not ka:
        return 0.0

    return len(ka & kb) / len(ka)


def parametric_duplicate_signal(
    emb_score: float,
    fuzzy_score: float,
    masked_score: float,
    signature_score: float,
    keyword_score: float,
    number_score: float,
) -> bool:
    if masked_score >= 0.78 and emb_score >= 0.70:
        return True

    if signature_score >= 0.82 and emb_score >= 0.68 and keyword_score >= 0.55:
        return True

    if masked_score >= 0.72 and signature_score >= 0.78 and keyword_score >= 0.60 and emb_score >= 0.68:
        return True

    if number_score == 0.0 and masked_score >= 0.75 and emb_score >= 0.74 and keyword_score >= 0.50:
        return True

    return False


def combined_score(
    emb_score: float,
    fuzzy_score: float,
    masked_score: float,
    signature_score: float,
    number_score: float,
    keyword_score: float,
) -> float:
    semantic_structural = (
        0.45 * emb_score
        + 0.25 * masked_score
        + 0.15 * signature_score
        + 0.15 * keyword_score
    )

    surface_numeric = (
        0.50 * emb_score
        + 0.20 * fuzzy_score
        + 0.20 * number_score
        + 0.10 * masked_score
    )

    return max(semantic_structural, surface_numeric)


def assign_verdict(
    exact_match: bool,
    emb_score: float,
    fuzzy_score: float,
    masked_score: float,
    signature_score: float,
    number_score: float,
    common_count: int,
    keyword_score: float,
    answer_match: bool,
    score: float,
) -> str:
    if exact_match:
        return "LIKELY_EXACT_DUPLICATE"

    if fuzzy_score >= 0.94:
        return "LIKELY_TEXT_NEAR_DUPLICATE"

    if parametric_duplicate_signal(
        emb_score=emb_score,
        fuzzy_score=fuzzy_score,
        masked_score=masked_score,
        signature_score=signature_score,
        keyword_score=keyword_score,
        number_score=number_score,
    ):
        return "LIKELY_PARAMETRIC_DUPLICATE"

    if number_score >= 0.70 and common_count >= 2 and emb_score >= 0.55:
        return "LIKELY_STRUCTURAL_DUPLICATE"

    if number_score >= 0.50 and emb_score >= 0.65 and keyword_score >= 0.40:
        return "REVIEW"

    if emb_score >= 0.78 and masked_score >= 0.65:
        return "REVIEW"

    if score >= 0.72:
        return "REVIEW"

    if answer_match and emb_score >= 0.65:
        return "REVIEW"

    return "LOW_RISK"


ds = load_dataset(HF_DATASET, HF_CONFIG, split=HF_SPLIT)

ds = ds.filter(
    lambda x: x.get("medium_pass_rate") is not None
    and x["medium_pass_rate"] > PASS_RATE_THRESHOLD
)

train = ds.to_pandas().reset_index(drop=True)
test = pd.read_csv(TEST_CSV).reset_index(drop=True)

required_test_cols = {"problem_id", "question"}
missing = required_test_cols - set(test.columns)
if missing:
    raise ValueError(f"отсутствуют обязательные колонки test: {missing}. текущие колонки: {list(test.columns)}")

train["check_text"] = (
    safe_col(train, "problem")
    + "\n\nORIGINAL_PROBLEM:\n"
    + safe_col(train, "original_problem")
)

test["check_text"] = test["question"].fillna("").astype(str)

if "answer" not in train.columns:
    train["answer"] = ""

if "answer" not in test.columns:
    test["answer"] = ""


train_texts_raw = train["check_text"].tolist()
test_texts_raw = test["check_text"].tolist()

train_norm = [normalize_math_text(x) for x in train_texts_raw]
test_norm = [normalize_math_text(x) for x in test_texts_raw]

train_masked = [mask_numbers(x) for x in train_texts_raw]
test_masked = [mask_numbers(x) for x in test_texts_raw]

train_signature = [compact_math_signature(x) for x in train_texts_raw]
test_signature = [compact_math_signature(x) for x in test_texts_raw]

model = SentenceTransformer(EMB_MODEL)

train_embed_texts = ["passage: " + normalize_for_embedding(x) for x in train_texts_raw]
test_embed_texts = ["query: " + normalize_for_embedding(x) for x in test_texts_raw]

train_emb = model.encode(
    train_embed_texts,
    batch_size=BATCH_SIZE,
    normalize_embeddings=True,
    show_progress_bar=True,
)

test_emb = model.encode(
    test_embed_texts,
    batch_size=BATCH_SIZE,
    normalize_embeddings=True,
    show_progress_bar=True,
)

train_emb = np.asarray(train_emb, dtype=np.float32)
test_emb = np.asarray(test_emb, dtype=np.float32)

emb_sims = np.matmul(test_emb, train_emb.T)


def top_indices_by_scores(scores: List[float], k: int) -> List[int]:
    if len(scores) == 0:
        return []
    arr = np.asarray(scores)
    k = min(k, len(arr))
    return np.argsort(-arr)[:k].tolist()


def number_hit_candidate_ids(test_text: str) -> List[int]:
    ids = []

    test_nums = set(extract_rare_numbers(test_text))
    if not test_nums:
        return ids

    for tr_i, train_text in enumerate(train_texts_raw):
        train_nums = set(extract_rare_numbers(train_text))
        common = test_nums & train_nums

        if len(common) >= 2:
            ids.append(tr_i)
            continue

        if any(x.isdigit() and int(x) >= 1000 for x in common):
            ids.append(tr_i)

    return ids


all_rows: List[Dict] = []

for ti in tqdm(range(len(test)), desc="test problems"):
    test_text = test_texts_raw[ti]

    top_embed_ids = np.argsort(-emb_sims[ti])[:TOPK_EMBEDDING].tolist()

    masked_scores_all = [
        fuzz.token_set_ratio(test_masked[ti], train_masked[j]) / 100.0
        for j in range(len(train))
    ]
    top_masked_ids = top_indices_by_scores(masked_scores_all, TOPK_MASKED_FUZZY)

    raw_scores_all = [
        fuzz.token_set_ratio(test_norm[ti], train_norm[j]) / 100.0
        for j in range(len(train))
    ]
    top_raw_ids = top_indices_by_scores(raw_scores_all, TOPK_RAW_FUZZY)

    top_num_ids = number_hit_candidate_ids(test_text)

    candidate_ids = sorted(set(top_embed_ids + top_masked_ids + top_raw_ids + top_num_ids))

    for tr_i in candidate_ids:
        train_text = train_texts_raw[tr_i]

        emb_score = float(emb_sims[ti, tr_i])
        fuzzy_score = raw_scores_all[tr_i]
        masked_score = masked_scores_all[tr_i]
        sig_score = signature_fuzzy(test_text, train_text)
        key_score = keyword_overlap_score(test_text, train_text)

        num_score, common_numbers, common_count = number_overlap_features(test_text, train_text)

        exact_match = exact_normalized_match(test_text, train_text)

        ans_match = answer_match_weak_flag(
            test.loc[ti, "answer"],
            train.loc[tr_i, "answer"],
        )

        score = combined_score(
            emb_score=emb_score,
            fuzzy_score=fuzzy_score,
            masked_score=masked_score,
            signature_score=sig_score,
            number_score=num_score,
            keyword_score=key_score,
        )

        verdict = assign_verdict(
            exact_match=exact_match,
            emb_score=emb_score,
            fuzzy_score=fuzzy_score,
            masked_score=masked_score,
            signature_score=sig_score,
            number_score=num_score,
            common_count=common_count,
            keyword_score=key_score,
            answer_match=ans_match,
            score=score,
        )

        should_save = (
            score >= MIN_SAVE_SCORE
            or verdict != "LOW_RISK"
            or tr_i in top_embed_ids[:10]
            or tr_i in top_masked_ids[:10]
            or tr_i in top_raw_ids[:10]
        )

        if not should_save:
            continue

        source_val = train.loc[tr_i, "source"] if "source" in train.columns else ""

        all_rows.append({
            "test_problem_id": test.loc[ti, "problem_id"],
            "train_row": tr_i,
            "verdict": verdict,
            "combined_score": round(score, 4),
            "embedding_score": round(emb_score, 4),
            "fuzzy_score": round(fuzzy_score, 4),
            "masked_fuzzy_score": round(masked_score, 4),
            "signature_fuzzy_score": round(sig_score, 4),
            "keyword_overlap_score": round(key_score, 4),
            "number_score": round(num_score, 4),
            "common_rare_numbers": common_numbers,
            "common_rare_number_count": common_count,
            "answer_match_weak_flag": ans_match,
            "test_answer": test.loc[ti, "answer"],
            "train_answer": train.loc[tr_i, "answer"],
            "source": source_val,
            "medium_pass_rate": train.loc[tr_i, "medium_pass_rate"] if "medium_pass_rate" in train.columns else None,
            "test_question": test_text,
            "train_problem": train.loc[tr_i, "problem"] if "problem" in train.columns else "",
            "train_original_problem": train.loc[tr_i, "original_problem"] if "original_problem" in train.columns else "",
        })


if not all_rows:
    print("кандидаты не сохранены")
    res = pd.DataFrame()
else:
    res = pd.DataFrame(all_rows)

    priority = {
        "LIKELY_EXACT_DUPLICATE": 0,
        "LIKELY_TEXT_NEAR_DUPLICATE": 1,
        "LIKELY_PARAMETRIC_DUPLICATE": 2,
        "LIKELY_STRUCTURAL_DUPLICATE": 3,
        "REVIEW": 4,
        "LOW_RISK": 5,
    }

    res["_priority"] = res["verdict"].map(priority).fillna(99).astype(int)

    res = res.sort_values(
        ["_priority", "test_problem_id", "combined_score"],
        ascending=[True, True, False],
    ).drop(columns=["_priority"])

    res.to_csv(OUT_ALL, index=False)

    suspicious = res[res["verdict"] != "LOW_RISK"].copy()
    suspicious.to_csv(OUT_SUSPICIOUS, index=False)

    top_by_test = (
        res.sort_values(["test_problem_id", "combined_score"], ascending=[True, False])
        .groupby("test_problem_id", as_index=False)
        .head(TOPK_PER_TEST_TO_SAVE)
    )
    top_by_test.to_csv(OUT_TOP_BY_TEST, index=False)

    print("\nсохранено:")
    print(" ", OUT_ALL)
    print(" ", OUT_SUSPICIOUS)
    print(" ", OUT_TOP_BY_TEST)

    print("\nподозрительные кандидаты:")
    if len(suspicious) == 0:
        print("нет кандидатов кроме LOW_RISK.")
    else:
        cols = [
            "test_problem_id",
            "train_row",
            "verdict",
            "combined_score",
            "embedding_score",
            "fuzzy_score",
            "masked_fuzzy_score",
            "signature_fuzzy_score",
            "keyword_overlap_score",
            "number_score",
            "common_rare_numbers",
            "answer_match_weak_flag",
            "source",
        ]
        print(suspicious[cols].to_string(index=False, max_rows=200))


def print_pair(row: pd.Series, max_chars: int = 1200):
    print("=" * 120)
    print(f"ТЕСТ {row['test_problem_id']}  vs  СТРОКА TRAIN {row['train_row']}")
    print(f"ВЕРДИКТ: {row['verdict']}")
    print(
        "оценки:",
        f"combined={row['combined_score']}",
        f"emb={row['embedding_score']}",
        f"fuzzy={row['fuzzy_score']}",
        f"masked={row['masked_fuzzy_score']}",
        f"signature={row['signature_fuzzy_score']}",
        f"keyword={row['keyword_overlap_score']}",
        f"num={row['number_score']}",
    )
    print("общие числа:", row["common_rare_numbers"])
    print("слабый флаг ответа:", row["answer_match_weak_flag"])
    print("\nТЕСТОВЫЙ ВОПРОС:")
    print(str(row["test_question"])[:max_chars])
    print("\nTRAIN ЗАДАЧА:")
    print(str(row["train_problem"])[:max_chars])
    print("\nORIGINAL ЗАДАЧА TRAIN:")
    print(str(row["train_original_problem"])[:max_chars])
    print()


if len(all_rows):
    print("\nдетали по подозрительным:")
    suspicious_for_print = res[res["verdict"] != "LOW_RISK"].head(30)

    if len(suspicious_for_print) == 0:
        suspicious_for_print = (
            res.sort_values(["test_problem_id", "combined_score"], ascending=[True, False])
            .groupby("test_problem_id", as_index=False)
            .head(1)
        )

    for _, row in suspicious_for_print.iterrows():
        print_pair(row)


if len(all_rows):
    likely = res[res["verdict"].str.startswith("LIKELY", na=False)]
    review = res[res["verdict"] == "REVIEW"]

    print("\nсводка:")
    print("LIKELY случаев:", len(likely))
    print("REVIEW случаев:", len(review))
    print("LOW_RISK сохраненных случаев:", int((res["verdict"] == "LOW_RISK").sum()))

    if len(likely):
        print("\nкраткая таблица LIKELY:")
        cols = [
            "test_problem_id",
            "train_row",
            "verdict",
            "combined_score",
            "embedding_score",
            "masked_fuzzy_score",
            "signature_fuzzy_score",
            "keyword_overlap_score",
            "number_score",
            "common_rare_numbers",
        ]
        print(likely[cols].to_string(index=False, max_rows=100))